# Credit Card Fraud Detection — Stage 1 Baseline & EDA
**Project:** Credit Card Fraud Detection (BTech Machine Learning Mini-Project)  
**Author:** Anish (Stage 1 Lead Engineer)  
**Collaborators:** Shubh (Stage 2: SVM & Random Forest), Dev (Stage 3: Evaluation & Deployment)  
**Dataset:** [Kaggle Credit Card Fraud Detection by Kartik Shenoy](https://www.kaggle.com/datasets/kartik2112/fraud-detection)

## 1. Setup and Module Imports
Import standard libraries and our shared modular package from `src/`.

In [ ]:
import os
import sys
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add root directory to sys.path
sys.path.append(os.path.abspath('..'))

from src.eda import perform_full_eda
from src.preprocessing import FraudPreprocessor, create_reproducible_splits
from src.models import tune_knn_classifier, tune_decision_tree_classifier, evaluate_model_performance
from src.visualization import plot_confusion_matrices, plot_roc_and_pr_curves

print('All modules loaded successfully.')

## 2. Exploratory Data Analysis (EDA)
We inspect the raw transaction data, verifying head(), tail(), info(), describe(), missing values, duplicates, and class imbalance.

In [ ]:
raw_train_path = '../data/raw/fraudTrain.csv'
df_raw = pd.read_csv(raw_train_path, nrows=50000)

print(f'Sample Shape: {df_raw.shape}')
print(f'Missing Values: {df_raw.isnull().sum().sum()}')
print(f'Duplicate Rows: {df_raw.duplicated().sum()}')
display(df_raw.head(3))

print('\nClass Imbalance:')
print(df_raw['is_fraud'].value_counts(normalize=True) * 100)

## 3. Load Leakage-Safe Preprocessed Data
We load the reproducible preprocessed feature matrices ($X_{train}, X_{val}, X_{test}$) generated by `src/preprocessing.py`.

In [ ]:
X_train, y_train = joblib.load('../data/processed/train_processed.joblib')
X_val, y_val = joblib.load('../data/processed/val_processed.joblib')
X_test, y_test = joblib.load('../data/processed/test_processed.joblib')

print(f'X_train shape: {X_train.shape} (Frauds: {y_train.sum()})')
print(f'X_val shape:   {X_val.shape} (Frauds: {y_val.sum()})')
print(f'X_test shape:  {X_test.shape} (Frauds: {y_test.sum()})')
print('\nEngineered Feature Columns (23 features):')
print(X_train.columns.tolist())

## 4. Model 1: K-Nearest Neighbors (KNN)
We evaluate distance-weighted KNN over $k \in [3, 5, 7, 9, 11]$ using SMOTE oversampling on the training set to prevent minority starvation.

In [ ]:
best_knn, best_k, knn_hist = tune_knn_classifier(
    X_train, y_train, X_val, y_val,
    k_values=[3, 5, 7, 9, 11],
    use_smote=True,
    random_state=42
)
print(f'Optimal KNN parameter: k = {best_k}')

## 5. Model 2: Decision Tree (ID3 / Information Gain vs CART / Gini)
We evaluate tree depths across both splitting criteria (Shannon Entropy vs Gini Impurity) with cost-sensitive class balancing.

In [ ]:
best_dt, best_dt_cfg, ent_hist, gini_hist = tune_decision_tree_classifier(
    X_train, y_train, X_val, y_val,
    depths=[3, 4, 5, 6, 8, 10, None],
    random_state=42
)
print(f'Optimal Decision Tree: {best_dt_cfg}')

## 6. Comparative Model Evaluation on Validation Set
Compute Accuracy, Precision, Recall, F1-Score, Confusion Matrices, ROC curves, and PR curves.

In [ ]:
knn_preds = best_knn.predict(X_val)
knn_proba = best_knn.predict_proba(X_val)[:, 1]
knn_metrics = evaluate_model_performance(y_val, knn_preds, knn_proba)

dt_preds = best_dt.predict(X_val)
dt_proba = best_dt.predict_proba(X_val)[:, 1]
dt_metrics = evaluate_model_performance(y_val, dt_preds, dt_proba)

summary_table = pd.DataFrame([
    {
        'Model': f'KNN (k={best_k}, SMOTE)',
        'Accuracy': f"{knn_metrics['accuracy']:.4f}",
        'Precision': f"{knn_metrics['precision']:.4f}",
        'Recall': f"{knn_metrics['recall']:.4f}",
        'F1-Score': f"{knn_metrics['f1']:.4f}",
        'ROC-AUC': f"{knn_metrics['roc_auc']:.4f}",
        'PR-AUC': f"{knn_metrics['pr_auc']:.4f}",
        'True Positives': knn_metrics['true_positives'],
        'False Positives': knn_metrics['false_positives']
    },
    {
        'Model': f"Decision Tree ({best_dt_cfg['criterion']})",
        'Accuracy': f"{dt_metrics['accuracy']:.4f}",
        'Precision': f"{dt_metrics['precision']:.4f}",
        'Recall': f"{dt_metrics['recall']:.4f}",
        'F1-Score': f"{dt_metrics['f1']:.4f}",
        'ROC-AUC': f"{dt_metrics['roc_auc']:.4f}",
        'PR-AUC': f"{dt_metrics['pr_auc']:.4f}",
        'True Positives': dt_metrics['true_positives'],
        'False Positives': dt_metrics['false_positives']
    }
])

display(summary_table)